In [8]:
import pandas as pd
import numpy as np

In [9]:
# Takes real data and adds artificial drift to simulate how fraud patterns change over time.
# drift_strength: 1.0 = normal drift, 2.0 = heavy drift

def create_drifted_data(df: pd.DataFrame, drift_strength: float = 1.0) -> pd.DataFrame:


    drifted = df.copy()

    np.random.seed(42)  

    # Shift Amount — fraudsters start doing higher value transactions
    # multiply by 1.4 = 40% increase in transaction amounts
    drifted['Amount'] = drifted['Amount'] * (1.4 * drift_strength) + \
                        np.random.normal(0, 8, len(drifted))

    # Shift V17, V12, V14 — your top features from feature importance
    # These are PCA components, shifting them = fraud pattern has changed
    drifted['V17'] = drifted['V17'] + np.random.normal(0.6 * drift_strength, 0.3, len(drifted))
    drifted['V12'] = drifted['V12'] + np.random.normal(0.5 * drift_strength, 0.2, len(drifted))
    drifted['V14'] = drifted['V14'] + np.random.normal(0.4 * drift_strength, 0.2, len(drifted))

    return drifted

In [10]:
# Returns data with NO drift — for showing the 'no drift' case in demo.Just adds tiny random noise that's not enough to trigger drift.

def create_normal_data(df: pd.DataFrame) -> pd.DataFrame:

    normal = df.copy()
    normal['Amount'] = normal['Amount'] + np.random.normal(0, 0.1, len(normal))
    return normal

In [12]:
if __name__ == '__main__':
    df = pd.read_csv('data/creditcard.csv')
    
    predictors = ['Time','V1','V2','V3','V4','V5','V6','V7','V8','V9','V10',
                  'V11','V12','V13','V14','V15','V16','V17','V18','V19',
                  'V20','V21','V22','V23','V24','V25','V26','V27','V28','Amount']

    split_point = int(len(df) * 0.7)
    
    # Reference period = first 70%
    reference_data = df.iloc[:split_point].copy()
    
    # Take a SAMPLE from reference period for normal data
    # This guarantees NO drift because it comes from same distribution
    normal_sample = reference_data.sample(n=10000, random_state=42).copy()
    
    # Add tiny noise — still same distribution
    normal_sample['Amount'] = normal_sample['Amount'] + \
                               np.random.normal(0, 0.05, len(normal_sample))
    
    # Drifted data = last 30% + strong artificial shift
    new_data = df.iloc[split_point:].copy()
    drifted_df = create_drifted_data(new_data, drift_strength=2.0)

    # Save both
    import os
    os.makedirs('data', exist_ok=True)
    
    normal_sample.to_csv('data/normal_sample.csv', index=False)
    drifted_df.to_csv('data/drifted_sample.csv', index=False)

    # Sanity check
    print("=" * 50)
    print("SANITY CHECK")
    print("=" * 50)
    print(f"Normal  Amount mean : {normal_sample['Amount'].mean():.2f}")
    print(f"Drifted Amount mean : {drifted_df['Amount'].mean():.2f}")
    print(f"Normal  V17 mean    : {normal_sample['V17'].mean():.4f}")
    print(f"Drifted V17 mean    : {drifted_df['V17'].mean():.4f}")
    print("=" * 50)
    print("✅ normal_sample.csv  → should show 🟢 NO DRIFT")
    print("✅ drifted_sample.csv → should show 🔴 DRIFT DETECTED")

SANITY CHECK
Normal  Amount mean : 93.87
Drifted Amount mean : 238.11
Normal  V17 mean    : 0.0207
Drifted V17 mean    : 1.1379
✅ normal_sample.csv  → should show 🟢 NO DRIFT
✅ drifted_sample.csv → should show 🔴 DRIFT DETECTED
